In [ ]:
import os

# Disable all WandB logging
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
trainer.train()


In [ ]:
# =========================
# 1. Install / imports
# =========================
!pip install -q -U transformers datasets accelerate scikit-learn

import os
import re
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

# Optional: keep logging quiet
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

set_seed(42)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# =========================
# 2. Upload dataset
# =========================
from google.colab import files
uploaded = files.upload()   # choose covmis_stance.csv

In [ ]:
# =========================
# 3. Load and preprocess
# =========================
df = pd.read_csv("covmis_stance.csv")

def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    return None

def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

# Filter for vaccine-related tweets
vaccine_keywords = ["vaccine", "pfizer", "moderna", "booster", "jab", "shot", "astrazeneca"]
mask = df["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df = df[mask].copy()

# Simplify labels + clean text
df["label_simplified"] = df["label"].apply(simplify_label)
df = df.dropna(subset=["label_simplified"])
df["text_cleaned"] = df["mis"].apply(clean_text)

print(df["label_simplified"].value_counts())
print("Total samples:", len(df))

# Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
id2label = {0: "misinfo", 1: "not_misinfo"}
df["label_id"] = df["label_simplified"].map(label2id)

# Keep only needed columns
df_model = df[["text_cleaned", "label_id"]].rename(columns={"label_id": "label"})
df_model.head()

In [ ]:
# =========================
# 4. Train/val/test split
# =========================
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.30,
    random_state=42,
    stratify=df_model["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# =========================
# 5. Convert to HF datasets
# =========================
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
# =========================
# 6. Tokenizer + tokenization
# =========================
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["text_cleaned"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned"])
val_ds = val_ds.remove_columns(["text_cleaned"])
test_ds = test_ds.remove_columns(["text_cleaned"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# =========================
# 7. Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

In [ ]:
# =========================
# 8. Model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# =========================
# 9. Training arguments
# =========================
args = TrainingArguments(
    output_dir="./ctbert_results",
    eval_strategy="epoch",          # works with your newer version path
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer.train()

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load pretrained model and tokenizer
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print("=== Classification Report ===")
print(classification_report(
    y_true,
    y_pred,
    target_names=["misinfo", "not_misinfo"],
    zero_division=0
))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))

In [ ]:
pred_output = trainer.predict(test_ds)

In [ ]:
# =========================
# 11. Train
# =========================
trainer.train()

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
from transformers import ProgressCallback

# remove the notebook callback that is causing the state error
trainer.remove_callback(NotebookProgressCallback)

# add normal progress callback instead
trainer.add_callback(ProgressCallback)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print("=== Classification Report ===")
print(classification_report(
    y_true,
    y_pred,
    target_names=["misinfo", "not_misinfo"],
    zero_division=0
))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.utils.notebook import NotebookProgressCallback

args = TrainingArguments(
    output_dir="./ctbert_results_clean",
    eval_strategy="no",
    save_strategy="no",
    logging_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
    disable_tqdm=True,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# make sure notebook callback is gone
trainer.remove_callback(NotebookProgressCallback)

In [ ]:
pred_output = trainer.predict(test_ds)

In [ ]:
# =========================
# 12. Evaluate on validation
# =========================
val_metrics = trainer.evaluate()
print("Validation metrics:")
print(val_metrics)

In [ ]:
# =========================
# 13. Final test predictions
# =========================
pred_output = trainer.predict(test_ds)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"], zero_division=0))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))

In [ ]:
# =========================
# 14. Save model
# =========================
trainer.save_model("./covid_ctbert_best")
tokenizer.save_pretrained("./covid_ctbert_best")
print("Saved to ./covid_ctbert_best")

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from transformers import AutoTokenizer

# 1️⃣ Load your pilot dataset
proc_dir = "./data_processed"
df = pd.read_csv(f"{proc_dir}/pilot_labels.csv")

# 2️⃣ Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

# 3️⃣ Split into train/test
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["label"]
)

# 4️⃣ Convert to Hugging Face Dataset format
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# 5️⃣ Tokenize
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text_cleaned"], truncation=True, padding="max_length", max_length=128
    )

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])

train_ds.set_format("torch")
test_ds.set_format("torch")

print("✅ train_ds and test_ds ready.")


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)
trainer.train()


In [ ]:
import pandas as pd

df = pd.read_csv(f"{raw_dir}/covmis_stance.csv")

# Vaccine keywords
vaccine_keywords = ["vaccine", "pfizer", "moderna", "booster", "jab", "shot", "astrazeneca"]

# Use the correct text column 'mis'
mask = df["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df_vaccine = df[mask].copy()

print("Original dataset size:", len(df))
print("Vaccine-related subset:", len(df_vaccine))
df_vaccine.head(5)



In [ ]:
# Simplify stances into two categories
def simplify_label(stance):
    stance = str(stance).lower()
    if stance in ["favor"]:      # supports the misinformation
        return "misinfo"
    elif stance in ["against", "neither"]:  # disagrees or neutral
        return "not_misinfo"
    else:
        return None

df_vaccine["label_simplified"] = df_vaccine["label"].apply(simplify_label)

# Drop missing or undefined labels
df_vaccine = df_vaccine.dropna(subset=["label_simplified"])

print(df_vaccine["label_simplified"].value_counts())
df_vaccine.head(5)



In [ ]:
import re

# 1️⃣ Clean tweet text
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))       # remove URLs
    text = re.sub(r"@\w+", "", text)               # remove mentions
    text = re.sub(r"#", "", text)                  # remove hashtag symbol
    text = re.sub(r"\s+", " ", text).strip()       # remove extra spaces
    return text.lower()

df_vaccine["text_cleaned"] = df_vaccine["mis"].apply(clean_text)

# 2️⃣ Balance classes (use equal counts)
n = min(df_vaccine["label_simplified"].value_counts().min(), 59)  # smallest class count
df_balanced = (
    df_vaccine.groupby("label_simplified", group_keys=False)
    .apply(lambda x: x.sample(n=n, random_state=42))
    .reset_index(drop=True)
)

# 3️⃣ Save the pilot dataset
df_balanced = df_balanced[["text_cleaned", "label_simplified"]]
print(df_balanced["label_simplified"].value_counts())

# 4️⃣ Write to Google Drive for later use
df_balanced.to_csv(f"{proc_dir}/pilot_labels.csv", index=False)

print(f"✅ Pilot dataset saved to: {proc_dir}/pilot_labels.csv")
df_balanced.head(5)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Load the cleaned pilot dataset
pilot_path = f"{proc_dir}/pilot_labels.csv"
df = pd.read_csv(pilot_path)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    df["text_cleaned"], df["label_simplified"], test_size=0.25, random_state=42, stratify=df["label_simplified"]
)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train logistic regression
model = LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 5, 10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [200, 400]
}

grid = GridSearchCV(LogisticRegression(class_weight='balanced'), params, cv=3, scoring='f1_macro')
grid.fit(X_train_vec, y_train)

print("Best Params:", grid.best_params_)
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_vec)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


In [ ]:
!pip install transformers torch tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
import pandas as pd

# Load pilot dataset
df = pd.read_csv(f"{proc_dir}/pilot_labels.csv")

# Encode labels
label2id = {'misinfo': 0, 'not_misinfo': 1}
df['label'] = df['label_simplified'].map(label2id)

# Split train/test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['label'])

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Load tokenizer & model
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")

# Training setup
args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

trainer.train()

# Evaluate
preds = trainer.predict(test_ds)
pred_labels = preds.predictions.argmax(-1)
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(test_df["label"], pred_labels, target_names=label2id.keys()))
print(confusion_matrix(test_df["label"], pred_labels))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch, pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
!pip install -U transformers


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
!pip install -U transformers torch datasets tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch, pandas as pd


In [ ]:
import torch, transformers
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


In [ ]:
args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

trainer.train()


In [ ]:
args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)


In [ ]:
args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # ✅ new name in v4.57+
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)
trainer.train()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "digitalepidemiologylab/covid-twitter-bert-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
trainer.train()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch, pandas as pd
from sklearn.model_selection import train_test_split

# 1️⃣ Load your pilot dataset
proc_dir = "./data_processed"  # adjust if needed
df = pd.read_csv(f"{proc_dir}/pilot_labels.csv")

# 2️⃣ Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

# 3️⃣ Split train/test
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["label"]
)

# 4️⃣ Convert to Hugging Face Dataset format
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# 5️⃣ Tokenize
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)
train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")

# 6️⃣ Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to("cuda" if torch.cuda.is_available() else "cpu")

# 7️⃣ Training setup (updated syntax for v4.57.1)
args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # ✅ correct for Transformers 4.57+
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)

# 8️⃣ Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

# 9️⃣ Train model
trainer.train()


In [ ]:
import os

print("Contents of current directory:", os.listdir("."))
if os.path.exists("data_processed"):
    print("Files in data_processed:", os.listdir("data_processed"))
else:
    print("❌ 'data_processed' folder not found.")


In [ ]:
import os, re, pandas as pd

raw_dir = "data_raw"
proc_dir = "data_processed"
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(proc_dir, exist_ok=True)

# Load the main dataset
df = pd.read_csv(f"{raw_dir}/covmis_stance.csv")

# Filter for vaccine-related tweets
vaccine_keywords = ["vaccine", "pfizer", "moderna", "booster", "jab", "shot", "astrazeneca"]
mask = df["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df_vaccine = df[mask].copy()

# Simplify stances into two categories
def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    else:
        return None

df_vaccine["label_simplified"] = df_vaccine["label"].apply(simplify_label)
df_vaccine = df_vaccine.dropna(subset=["label_simplified"])

# Clean text
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

df_vaccine["text_cleaned"] = df_vaccine["mis"].apply(clean_text)

# Balance dataset
n = min(df_vaccine["label_simplified"].value_counts().min(), 59)
df_balanced = (
    df_vaccine.groupby("label_simplified", group_keys=False)
    .apply(lambda x: x.sample(n=n, random_state=42))
    .reset_index(drop=True)
)
df_balanced = df_balanced[["text_cleaned", "label_simplified"]]

# Save pilot dataset
df_balanced.to_csv(f"{proc_dir}/pilot_labels.csv", index=False)
print(f"✅ Pilot dataset created at: {proc_dir}/pilot_labels.csv")


In [ ]:
!ls -R


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose covmis_stance.csv from your device


In [ ]:
import shutil
shutil.move("covmis_stance.csv", "data_raw/covmis_stance.csv")


In [ ]:
!ls -lh data_raw


In [ ]:
import pandas as pd

df = pd.read_csv("data_raw/covmis_stance.csv")
print(df.shape)
print(df.columns.tolist())
df.head()


In [ ]:
import re, os

raw_dir = "data_raw"
proc_dir = "data_processed"

os.makedirs(proc_dir, exist_ok=True)

# Filter for vaccine-related tweets
vaccine_keywords = ["vaccine", "pfizer", "moderna", "booster", "jab", "shot", "astrazeneca"]
mask = df["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df_vaccine = df[mask].copy()

# Simplify stance labels
def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    return None

df_vaccine["label_simplified"] = df_vaccine["label"].apply(simplify_label)
df_vaccine = df_vaccine.dropna(subset=["label_simplified"])

# Clean text
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

df_vaccine["text_cleaned"] = df_vaccine["mis"].apply(clean_text)

# Balance and save pilot dataset
n = min(df_vaccine["label_simplified"].value_counts().min(), 59)
df_balanced = df_vaccine.groupby("label_simplified", group_keys=False).apply(lambda x: x.sample(n=n, random_state=42)).reset_index(drop=True)
df_balanced = df_balanced[["text_cleaned", "label_simplified"]]
df_balanced.to_csv(f"{proc_dir}/pilot_labels.csv", index=False)

print("✅ Pilot dataset saved at:", f"{proc_dir}/pilot_labels.csv")


In [ ]:
!ls data_processed


In [ ]:
trainer.train()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch, pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
proc_dir = "./data_processed"
df = pd.read_csv(f"{proc_dir}/pilot_labels.csv")

label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])

train_ds.set_format("torch")
test_ds.set_format("torch")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to("cuda" if torch.cuda.is_available() else "cpu")

args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ✅ correct key for v4.57.1
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=5,
    learning_rate=2e-5,
    weight_decay=0.01
)


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)


In [ ]:
trainer.train()


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
trainer.train()


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch, pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
trainer.train()


In [ ]:
!ls data_raw


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
print(confusion_matrix(y_true, y_pred))


In [ ]:
trainer.save_model("./covid_misinfo_model")
tokenizer.save_pretrained("./covid_misinfo_model")


In [ ]:
df_full = pd.read_csv("data_raw/covmis_stance.csv")
# Repeat same filtering & label simplification but skip the .sample() step


In [ ]:
args.num_train_epochs = 5
args.learning_rate = 3e-5


In [ ]:
from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
df_full = pd.read_csv("data_raw/covmis_stance.csv")


In [ ]:
import re, pandas as pd

def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    return None

def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

# Apply filters
vaccine_keywords = ["vaccine","pfizer","moderna","booster","jab","shot","astrazeneca"]
mask = df_full["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df_full = df_full[mask].copy()

# Label + clean text
df_full["label_simplified"] = df_full["label"].apply(simplify_label)
df_full = df_full.dropna(subset=["label_simplified"])
df_full["text_cleaned"] = df_full["mis"].apply(clean_text)
print(df_full["label_simplified"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_full, test_size=0.15, random_state=42, stratify=df_full["label_simplified"]
)


In [ ]:
from datasets import Dataset

label2id = {"misinfo": 0, "not_misinfo": 1}
train_df["label"] = train_df["label_simplified"].map(label2id)
test_df["label"] = test_df["label_simplified"].map(label2id)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

args = TrainingArguments(
    output_dir="./results_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


In [ ]:
preds = trainer.predict(test_ds)
from sklearn.metrics import classification_report
print(classification_report(test_df["label"], preds.predictions.argmax(-1)))


In [ ]:
trainer.save_model("./covid_misinfo_model")
tokenizer.save_pretrained("./covid_misinfo_model")


In [ ]:
import pandas as pd, re, os

raw_dir = "data_raw"
proc_dir = "data_processed"
os.makedirs(proc_dir, exist_ok=True)

df = pd.read_csv(f"{raw_dir}/covmis_stance.csv")
print(df.shape)
df.head(3)


In [ ]:
# Vaccine keywords
vaccine_keywords = ["vaccine", "pfizer", "moderna", "booster", "jab", "shot", "astrazeneca"]
mask = df["mis"].astype(str).str.lower().str.contains("|".join(vaccine_keywords), na=False)
df_full = df[mask].copy()

# Simplify labels
def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    return None

# Clean text
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

df_full["label_simplified"] = df_full["label"].apply(simplify_label)
df_full["text_cleaned"] = df_full["mis"].apply(clean_text)
df_full = df_full.dropna(subset=["label_simplified"])

print(df_full["label_simplified"].value_counts())
df_full.to_csv(f"{proc_dir}/full_labels.csv", index=False)
print("✅ Saved expanded dataset:", f"{proc_dir}/full_labels.csv")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd

# Load full dataset
df = pd.read_csv("data_processed/full_labels.csv")

label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

# Split
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["label"])

# Convert to HF dataset
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

# Tokenize
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_fn(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds  = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")

# Model & training setup
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(
    output_dir="./results_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.train()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
print(confusion_matrix(y_true, y_pred))


In [ ]:
from nlpaug.augmenter.word import SynonymAug
aug = SynonymAug(aug_src='wordnet')
df_aug = df_full.copy()
df_aug['text_cleaned'] = df_aug['text_cleaned'].apply(lambda t: aug.augment(t))


In [ ]:
!pip install nlpaug


In [ ]:
from nlpaug.augmenter.word import SynonymAug

aug = SynonymAug(aug_src='wordnet')
df_aug = df_full.copy()
df_aug['text_cleaned'] = df_aug['text_cleaned'].apply(lambda t: aug.augment(t))


In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')


In [ ]:
nltk.download('punkt')


In [ ]:
from nlpaug.augmenter.word import SynonymAug

aug = SynonymAug(aug_src='wordnet')
df_aug = df_full.copy()
df_aug['text_cleaned'] = df_aug['text_cleaned'].apply(lambda t: aug.augment(t))


In [ ]:
# Combine original + augmented
df_expanded = pd.concat([df_full, df_aug], ignore_index=True)
df_expanded.to_csv("data_processed/full_augmented.csv", index=False)
print("✅ Augmented dataset saved:", len(df_expanded), "samples")

# Reload into Hugging Face Dataset and retrain
from datasets import Dataset
train_df, test_df = train_test_split(df_expanded, test_size=0.15, random_state=42, stratify=df_expanded["label_simplified"])

label2id = {"misinfo": 0, "not_misinfo": 1}
train_df["label"] = train_df["label_simplified"].map(label2id)
test_df["label"] = test_df["label_simplified"].map(label2id)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

def tokenize_function(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")

# Re-train with same parameters
trainer.train()


In [ ]:
# Fix augmented output to ensure plain text strings
df_aug["text_cleaned"] = df_aug["text_cleaned"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)


In [ ]:
df_expanded = pd.concat([df_full, df_aug], ignore_index=True)
df_expanded.to_csv("data_processed/full_augmented.csv", index=False)
print("✅ Augmented dataset saved:", len(df_expanded), "samples")


In [ ]:
df_aug["text_cleaned"].apply(type).value_counts()


In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load augmented data
df_expanded = pd.read_csv("data_processed/full_augmented.csv")

# Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
df_expanded["label"] = df_expanded["label_simplified"].map(label2id)

# Split dataset
train_df, test_df = train_test_split(
    df_expanded, test_size=0.15, random_state=42, stratify=df_expanded["label"]
)

# Convert to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Tokenize
def tokenize_function(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)
train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])
train_ds.set_format("torch")
test_ds.set_format("torch")

# Re-train using your same model and args
trainer.train()

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix
preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)
print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
print(confusion_matrix(y_true, y_pred))


In [ ]:
from torch.nn import CrossEntropyLoss
loss_fn = CrossEntropyLoss(weight=torch.tensor([1.3, 1.0]).to(device))


In [ ]:
import torch
from torch.nn import CrossEntropyLoss

# Define device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create class-weighted loss function
loss_fn = CrossEntropyLoss(weight=torch.tensor([1.3, 1.0]).to(device))


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1️⃣ Train using the weighted loss
trainer.train()

# 2️⃣ Evaluate on test set
preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

# 3️⃣ Display evaluation metrics
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_true, y_pred))


In [ ]:
!pip install transformers sentencepiece datasets torchtext googletrans==4.0.0-rc1


In [ ]:
 from googletrans import Translator
translator = Translator()

def back_translate(text, lang='de'):  # English → German → English
    try:
        return translator.translate(translator.translate(text, dest=lang).text, dest='en').text
    except:
        return text

df_bt = df_full.copy()
df_bt["text_cleaned"] = df_bt["text_cleaned"].apply(back_translate)
df_bt["label_simplified"] = df_bt["label_simplified"]
df_expanded_bt = pd.concat([df_full, df_bt], ignore_index=True)
df_expanded_bt.to_csv("data_processed/full_backtranslated.csv", index=False)
print("✅ Back-translated dataset saved:", len(df_expanded_bt))


In [ ]:
import pandas as pd

# Load your full dataset (the same one you saved earlier)
df_full = pd.read_csv("data_processed/full_labels.csv")

print("Loaded dataset:", df_full.shape)
print(df_full.head(2))


In [ ]:
from googletrans import Translator
translator = Translator()

def back_translate(text, lang='de'):
    try:
        return translator.translate(translator.translate(text, dest=lang).text, dest='en').text
    except:
        return text

df_bt = df_full.copy()
df_bt["text_cleaned"] = df_bt["text_cleaned"].apply(back_translate)
df_bt["label_simplified"] = df_bt["label_simplified"]
df_expanded_bt = pd.concat([df_full, df_bt], ignore_index=True)
df_expanded_bt.to_csv("data_processed/full_backtranslated.csv", index=False)
print("✅ Back-translated dataset saved:", len(df_expanded_bt))


In [ ]:
args = TrainingArguments(
    output_dir="./results_tuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,                  # gradual warmup
    lr_scheduler_type="cosine",        # smooth decay
    num_train_epochs=10,               # more epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
from transformers import Trainer, TrainingArguments


In [ ]:
args = TrainingArguments(
    output_dir="./results_tuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,                  # gradual warmup
    lr_scheduler_type="cosine",        # smooth decay
    num_train_epochs=10,               # more epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
eval_strategy="epoch",


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./results_tuned",
    eval_strategy="epoch",         # ✅ correct for your version
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
model_name = "jhu-clsp/health-covid-bert"


In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name, num_labels=2, hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)


In [ ]:
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"


In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification

config = AutoConfig.from_pretrained(model_name, num_labels=2, hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the processed or augmented dataset
df = pd.read_csv("data_processed/full_augmented.csv")  # or full_backtranslated.csv / full_labels.csv

# Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

# Split into train/test
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

# Convert to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Tokenize
def tokenize_function(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])

train_ds.set_format("torch")
test_ds.set_format("torch")

print("✅ train_ds and test_ds recreated:", len(train_ds), "training samples,", len(test_ds), "test samples")


In [ ]:
from transformers import AutoTokenizer

# Choose your model name
model_name = "digitalepidemiologylab/covid-twitter-bert-v2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✅ Tokenizer loaded successfully")


In [ ]:
def tokenize_function(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd

# Load the processed or augmented dataset
df = pd.read_csv("data_processed/full_augmented.csv")  # or full_backtranslated.csv / full_labels.csv

# Encode labels
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)

# Split into train/test
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

# Convert to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Tokenize
def tokenize_function(batch):
    return tokenizer(batch["text_cleaned"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text_cleaned", "label_simplified"])
test_ds = test_ds.remove_columns(["text_cleaned", "label_simplified"])

train_ds.set_format("torch")
test_ds.set_format("torch")

print("✅ train_ds and test_ds recreated:", len(train_ds), "training samples,", len(test_ds), "test samples")


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
!rm -rf ~/.config/wandb ~/.cache/wandb


In [ ]:
args = TrainingArguments(
    output_dir="./results_tuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    report_to="none",        # 👈 prevents wandb or tensorboard logging
    logging_dir=None,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
!pip install -U transformers


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./results_tuned",
    evaluation_strategy="epoch",   # ✅ works in latest versions
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    report_to="none",
    logging_dir=None,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
args = TrainingArguments(
    output_dir="./results_tuned",
    eval_strategy="epoch",        # 👈 works on older versions (< 4.20)
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    report_to="none",
    logging_dir=None,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from accelerate.state import AcceleratorState

# Reset the current accelerate state
AcceleratorState._reset_state()

# Recreate Trainer cleanly
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
preds = trainer.predict(test_ds)


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

AcceleratorState._reset_state()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

preds = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["misinfo", "not_misinfo"]))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
import gc, torch
from accelerate.state import AcceleratorState

# Clean up previous states
gc.collect()
torch.cuda.empty_cache()
AcceleratorState._reset_state()


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

args = TrainingArguments(
    output_dir="./results_tuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./results_tuned",
    eval_strategy="epoch",           # ✅ older versions use eval_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,                # gradual warmup
    lr_scheduler_type="cosine",      # smooth decay
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"                 # disables wandb/tensorboard
)


In [ ]:
from accelerate.state import AcceleratorState
AcceleratorState._reset_state()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.train()


In [ ]:
from googletrans import Translator
translator = Translator()

def back_translate(text, lang='de'):
    try:
        return translator.translate(translator.translate(text, dest=lang).text, dest='en').text
    except:
        return text


In [ ]:
df_aug_syn = df_full.copy()
df_aug_syn["text_cleaned"] = df_aug_syn["text_cleaned"].apply(lambda t: " ".join(aug.augment(t)))
df_bt = df_full.copy()
df_bt["text_cleaned"] = df_bt["text_cleaned"].apply(back_translate)

df_expanded = pd.concat([df_full, df_aug_syn, df_bt], ignore_index=True)
df_expanded.to_csv("data_processed/full_expanded.csv", index=False)


In [ ]:
# Re-import and initialize synonym augmenter
from nlpaug.augmenter.word import SynonymAug

# Create augmenter (WordNet-based synonym replacement)
aug = SynonymAug(aug_src='wordnet')


In [ ]:
df_aug_syn = df_full.copy()
df_aug_syn["text_cleaned"] = df_aug_syn["text_cleaned"].apply(lambda t: " ".join(aug.augment(t)))

df_bt = df_full.copy()
df_bt["text_cleaned"] = df_bt["text_cleaned"].apply(back_translate)

df_expanded = pd.concat([df_full, df_aug_syn, df_bt], ignore_index=True)
df_expanded.to_csv("data_processed/full_expanded.csv", index=False)

print("✅ Expanded dataset saved:", len(df_expanded))


In [ ]:
df_aug_syn = df_full.copy()
df_aug_syn["text_cleaned"] = df_aug_syn["text_cleaned"].apply(lambda t: " ".join(aug.augment(t)))
df_bt = df_full.copy()
df_bt["text_cleaned"] = df_bt["text_cleaned"].apply(back_translate)

df_expanded = pd.concat([df_full, df_aug_syn, df_bt], ignore_index=True)
df_expanded.to_csv("data_processed/full_expanded.csv", index=False)


In [ ]:
args.num_train_epochs = 8
args.load_best_model_at_end = True
args.metric_for_best_model = "eval_loss"
args.save_strategy = "epoch"


In [ ]:
callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]


In [ ]:
from torch.nn import CrossEntropyLoss
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = CrossEntropyLoss(weight=torch.tensor([1.3, 1.0]).to(device))


In [ ]:
for lr in [3e-5, 2e-5, 1e-5]:
    args.learning_rate = lr
    trainer.train()


In [ ]:
learning_rate = 2e-5
num_train_epochs = 8
eval_strategy = "epoch"
save_strategy = "epoch"
EarlyStoppingCallback(patience=2)


In [ ]:
trainer.save_model("./covid_misinfo_best")
tokenizer.save_pretrained("./covid_misinfo_best")


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
!pip install -U transformers datasets nlpaug sentencepiece googletrans==4.0.0-rc1


In [ ]:
df = pd.read_csv("data_processed/full_expanded.csv")  # or whichever you last used
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)


In [ ]:
import pandas as pd



In [ ]:
df = pd.read_csv("data_processed/full_expanded.csv")  # or whichever you last used
label2id = {"misinfo": 0, "not_misinfo": 1}
df["label"] = df["label_simplified"].map(label2id)


In [ ]:
!ls data_processed


In [ ]:
import os, pandas as pd
os.makedirs("data_processed", exist_ok=True)

# Recreate your base dataset
df_full = pd.read_csv("data_raw/covmis_stance.csv")

# Simplify and clean again (same as before)
def simplify_label(stance):
    stance = str(stance).lower()
    if stance == "favor":
        return "misinfo"
    elif stance in ["against", "neither"]:
        return "not_misinfo"
    return None

def clean_text(text):
    import re
    text = re.sub(r"http\S+", "", str(text))
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

df_full["label_simplified"] = df_full["label"].apply(simplify_label)
df_full["text_cleaned"] = df_full["mis"].apply(clean_text)
df_full = df_full.dropna(subset=["label_simplified"])

# Save again
df_full.to_csv("data_processed/full_labels.csv", index=False)
print("✅ Saved:", len(df_full), "samples -> data_processed/full_labels.csv")


In [ ]:
/content/covmis_stance.csv


In [ ]:
df_full = pd.read_csv("/content/covmis_stance.csv")
